# Name That LLM: GLM-5.3 likelihood features for train.csv, on a TPU VM's CPUs

The same scoring as the GPU notebook (24 likelihood features per passage under GLM-5.3 and GLM-5.3-Flash, written to `glm_ll/`), for the TPU VM v5e-8 accelerator. llama.cpp cannot use the TPU chips, so this notebook builds `ntl-segrun` for the VM's host CPUs and runs it there with one thread per core. Needs **internet on** (to clone llama.cpp).

`MODE`: `probe` checks the machine and the inputs only; `timing` scores the first 40 train passages (the same ones as the GPU timing run) and forecasts the full run; `full` scores all of train.csv, or parts of it in order with `SHARDS = [(k, n), ...]` (part k of n: passages k, k+n, k+2n, ...). Each part is saved as soon as it finishes, so a session that ends early keeps the parts it completed. `MODELS` picks the models of this session.

**Caveat.** Kaggle staff have said that a TPU session whose code uses only the CPUs for about 30 minutes is stopped (Kaggle discussion 607202, 2025-10-28). This notebook never touches the TPU, so expect that stop; it does not try to get around it.

The scoring pipeline (`ntl-segrun` and its llama.cpp patch, `validate.sh`, `run_pipeline.py`, `features.py`, `segio.py`) is José Padilla's, from *ntl-glm-likelihood-train* and *0.689 | Real GLM-5.3 Likelihood on 2xT4*; the segment GGUFs are jamesliebel's mirrors of jinghengwu's quantization of Z.ai's open weights. This notebook only builds them for a CPU and runs them over train.csv.

In [ ]:
MODE = 'timing'  # 'probe' | 'timing' | 'full'
MODELS = ('flash', 'glm53')  # the models scored in this session, in this order
SHARDS = [(0, 1)]  # full mode: parts in order; merge them offline
# from the timing run (builder --forecast): model -> (fixed s, compute s for all of train.csv)
FORECAST = {}
THREADS = 0  # 0: one per physical core
SEQ_BATCH = 8  # texts per llama_decode, as in the GPU runs
SESSION_LIMIT_S = 8.5 * 3600  # stop starting work after this (TPU sessions end at 9 h)
PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>'
MAX_TEXT_TOKENS = 160
TIMING_ROWS = 40
LLAMA_TAG = 'b12247'
SEGMENTS = {"jamesliebel/glm53-iq3xxs-segs-a-mirror": [["glm53_iq3xxs-seg00-of-19-L0-6.gguf", 12753355264], ["glm53_iq3xxs-seg01-of-19-L6-10.gguf", 15651714688], ["glm53_iq3xxs-seg02-of-19-L10-14.gguf", 14583346912], ["glm53_iq3xxs-seg03-of-19-L14-18.gguf", 14583346912], ["glm53_iq3xxs-seg04-of-19-L18-22.gguf", 14583346912], ["glm53_iq3xxs-seg05-of-19-L22-26.gguf", 14583346912], ["glm53_iq3xxs-seg06-of-19-L26-30.gguf", 14583346912], ["glm53_iq3xxs-seg07-of-19-L30-34.gguf", 14583346912], ["glm53_iq3xxs-seg08-of-19-L34-38.gguf", 14583346912], ["glm53_iq3xxs-seg09-of-19-L38-42.gguf", 14583346912]], "jamesliebel/glm53-iq3xxs-segs-b-mirror": [["glm53_iq3xxs-seg10-of-19-L42-46.gguf", 14583346912], ["glm53_iq3xxs-seg11-of-19-L46-50.gguf", 14583346912], ["glm53_iq3xxs-seg12-of-19-L50-54.gguf", 14583346912], ["glm53_iq3xxs-seg13-of-19-L54-58.gguf", 14583346912], ["glm53_iq3xxs-seg14-of-19-L58-62.gguf", 14583346912], ["glm53_iq3xxs-seg15-of-19-L62-66.gguf", 14583346912], ["glm53_iq3xxs-seg16-of-19-L66-70.gguf", 14583346912], ["glm53_iq3xxs-seg17-of-19-L70-74.gguf", 14583346912], ["glm53_iq3xxs-seg18-of-19-L74-78.gguf", 16345433920]], "jamesliebel/glm53-flash-iq3xxs-segs-mirror": [["flash_iq3xxs-seg00-of-06-L0-8.gguf", 14879704352], ["flash_iq3xxs-seg01-of-06-L8-16.gguf", 22864262976], ["flash_iq3xxs-seg02-of-06-L16-24.gguf", 21821175680], ["flash_iq3xxs-seg03-of-06-L24-32.gguf", 21821175680], ["flash_iq3xxs-seg04-of-06-L32-40.gguf", 21821175680], ["flash_iq3xxs-seg05-of-06-L40-45.gguf", 14407253632]]}
SEGRUN_SHA256 = {
 "llama-cpp-b12247-ntl.patch": "98d54fd81f292c605376ca6d106dfe7fa401bf367d4a8b29e83e1ab55837c8cd",
 "validate.sh": "6ef51f6f890b44054436a64f6cb6290a4f2bad531bfc850f978dc3141f49a43d",
 "slice_gguf.py": "af4d5b98486334b557588da12de9f55c6672fc6767f483521c18fea0d3feeb86",
 "compare.py": "591ddf0600744b16372502090158c8a43661f319f739df96949301ceac900cf0"
}

import base64
import glob
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

T0 = time.time()


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-6000:], r.stderr[-3000:], flush=True)
    if check:
        assert r.returncode == 0, f'failed ({r.returncode}): {cmd}'
    return r


# ---- the machine: a TPU v5e-8 VM has 224 vCPUs (2 x 56-core AMD EPYC); a push can fall back to 4 ----
sh('nproc; lscpu | head -30; free -g; df -h / /kaggle/working /kaggle/temp /tmp /dev/shm 2>/dev/null; '
   'cat /sys/fs/cgroup/cpu.max /sys/fs/cgroup/memory.max 2>/dev/null; ls /dev/accel* /dev/vfio 2>/dev/null; '
   'mount | grep /kaggle/input | head -3; which git gcc g++ cmake make; gcc --version | head -1', check=False)
print('image', os.environ.get('KAGGLE_DOCKER_IMAGE'), flush=True)
assert len(os.sched_getaffinity(0)) >= 200, (
    f'{len(os.sched_getaffinity(0))} CPUs: not a TPU VM host (did the push fall back to a CPU machine?)')
cpu = sh('lscpu -p=CORE,SOCKET', check=False).stdout
PHYSICAL = len({line for line in cpu.splitlines() if line and not line.startswith('#')}) or os.cpu_count()
FLAGS = set(re.search(r'^flags\s*:(.*)$', open('/proc/cpuinfo').read(), re.M).group(1).split())
print(f'{os.cpu_count()} logical CPUs, {PHYSICAL} physical cores; AVX2 {"avx2" in FLAGS}, '
      f'AVX-512 {"avx512f" in FLAGS}, AVX-512 VNNI {"avx512_vnni" in FLAGS}, AMX {"amx_tile" in FLAGS}')
assert {'avx2', 'fma', 'f16c'} <= FLAGS, 'no AVX2/FMA/F16C: ggml would be very slow here'
mem = int(re.search(r'MemAvailable:\s+(\d+) kB', open('/proc/meminfo').read()).group(1)) / 2**20
print(f'{mem:.0f} GiB RAM available', flush=True)
assert mem >= 100, 'too little RAM to keep a segment in the page cache'
# No OMP_PROC_BIND/OMP_PLACES: libgomp would pin ntl-segrun's main thread to one core, and the
# per-text scoring threads it starts (std::thread, not OpenMP) would all inherit that core.
THREADS = THREADS or PHYSICAL

# work folder: the local disk with the most free space (the state between segments goes there;
# never /kaggle/working, which is ~20 GB and becomes the saved output)
roots = [Path(p) for p in ('/kaggle/temp', '/tmp', '/') if os.path.isdir(p) and os.access(p, os.W_OK)]
ROOT = max(roots, key=lambda p: shutil.disk_usage(p).free)
WORK = ROOT / 'glm_ll'                 # passage lists and the state between segments
BASE = Path('/tmp/ntl')                # validate.sh expects src/segrun and third_party/llama.cpp side by side
SRC = BASE / 'src' / 'segrun'
LLAMA = BASE / 'third_party' / 'llama.cpp'
BIN = BASE / 'bin'
OUT = Path('/kaggle/working/glm_ll')
for d in (WORK, SRC, BIN, OUT):
    d.mkdir(parents=True, exist_ok=True)
print('work folder', WORK, f'{shutil.disk_usage(ROOT).free / 2**30:.0f} GiB free', flush=True)
# the f32 state of a full Flash run is 22.9 GB (GLM-5.3: 8.6 GB); less room means more chunks, and every
# chunk reads all the segments again
if shutil.disk_usage(ROOT).free < 35e9:
    print('WARNING: under 35 GB free for the state: a full Flash run will need several chunks', flush=True)

# ---- the inputs: every segment present, with its exact size ----
TRAIN = sorted(glob.glob('/kaggle/input/**/name-that-llm/train.csv', recursive=True))
assert TRAIN, 'competition data not attached'
TRAIN = TRAIN[0]
FLASH, GLM53 = [], []
for ref, files in SEGMENTS.items():
    for name, size in files:
        hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
        assert hits, f'{name} missing: attach {ref}'
        got = os.path.getsize(hits[0])
        assert got == size, f'{hits[0]}: {got} bytes, expected {size}'
        (FLASH if name.startswith('flash_') else GLM53).append(hits[0])
FLASH.sort()
GLM53.sort()
assert len(FLASH) == 6 and len(GLM53) == 19
print(f'{len(pd.read_csv(TRAIN))} train passages; 25 segments found with the right sizes '
      f'({sum(os.path.getsize(p) for p in FLASH + GLM53) / 1e9:.0f} GB)', flush=True)
# read speed of the inputs (4 GiB of one segment; the first read of each segment is a cold one)
sh(f'dd if={FLASH[1]} of=/dev/null bs=16M count=256 2>&1 | tail -1', check=False)


In [ ]:
SEGRUN_B64 = {
 "llama-cpp-b12247-ntl.patch": "...",
 "validate.sh": "IyEvdXNyL2Jpbi9lbnYgYmFzaAojIFZhbGlkYXRlIG50bC1zZWdydW46IGEgR0xNNS1OZXh0IG1vZGVsIHJ1biBieSBsYXllciBzZWdtZW50cyBtdXN0IGdpdmUgdGhlIHNhbWUgc2NvcmVzIGFzIHJ1biB3aG9sZS4KIyAgIHNyYy9zZWdydW4vdmFsaWRhdGUuc2ggQlVJTERfQklOX0RJUiBQWVRIT04gW05HTF0gW0FSQ0g9Z2xtNS1uZXh0fGdsbS1kc2FdCiMgQnVpbGRzIGEgcmFuZG9tIDEyLWxheWVyIG1vZGVsIG9mIEFSQ0ggKHRlc3QtbGxhbWEtYXJjaHMsIGluZGV4ZXIgdG9wLWsgNjQ6IGFib3ZlIHRoZSBsb25nZXN0IHRleHQsIGJlbG93IHRoZQojIEtWIGNlbGxzIG9mIGEgdW5pZmllZCBjYWNoZSB3aXRoIHNldmVyYWwgdGV4dHMpLCBjdXRzIGl0IGludG8gMyBzZWdtZW50cwojIChzbGljZV9nZ3VmLnB5KSwgcnVucyB0aGUgd2hvbGUgbW9kZWwgYW5kIHRoZSBzZWdtZW50IHBpcGVsaW5lIHdpdGggZjMyL2YxNi9iZjE2IHN0YXRlcyBhbmQgc2VxLWJhdGNoIDEgYW5kIDQsCiMgdGhlbiBjb21wYXJlcyAoY29tcGFyZS5weSkuCnNldCAtZXVvIHBpcGVmYWlsCkJJTj0kezE6P2J1aWxkIGJpbiBkaXJ9ClBZPSR7Mjo/cHl0aG9ufQpOR0w9JHszOi0wfQpBUkNIPSR7NDotZ2xtNS1uZXh0fQpIRVJFPSQoY2QgIiQoZGlybmFtZSAiJDAiKSIgJiYgcHdkKQpUPSR7TlRMX1RFU1RfRElSOi0kSEVSRS8uLi8uLi9vdXRwdXRzL3NlZ3J1bl90ZXN0fS8kQVJDSApybSAtcmYgIiRUIjsgbWtkaXIgLXAgIiRUIgpOVExfVEVTVF9HTE01X05MQVlFUj0xMiBOVExfVEVTVF9JTkRFWEVSX1RPUEs9JHtOVExfVEVTVF9JTkRFWEVSX1RPUEs6LTY0fSAiJEJJTi90ZXN0LWxsYW1hLWFyY2hzIiAtYSAiJEFSQ0giIC1vICIkVCIgPiAiJFQvZ2VuLmxvZyIgMj4mMQoiJFBZIiAtSSAiJEhFUkUvc2xpY2VfZ2d1Zi5weSIgIiRULyRBUkNILW1vZS5nZ3VmIiAiJFQvc2VncyIgMCw0LDgsMTIgPiAiJFQvc2xpY2UubG9nIgoiJFBZIiAtSSAtYyAiCmltcG9ydCBzeXM7IHN5cy5wYXRoLmluc2VydCgwLCAnJEhFUkUnKTsgaW1wb3J0IHNlZ2lvLCBudW1weSBhcyBucApybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMCkKc2VxcyA9IFsocm5nLmludGVnZXJzKDAsIDEyOCwgc2l6ZT1pbnQoTCkpLnRvbGlzdCgpLCBpbnQocCkpIGZvciBMLCBwIGluIHppcChybmcuaW50ZWdlcnMoOCwgNjAsIHNpemU9MTYpLCBybmcuaW50ZWdlcnMoMSwgNiwgc2l6ZT0xNikpXQpzZWdpby53cml0ZV90b2tlbnMoJyRUL3Rvay5iaW4nLCBzZXFzKQoiCnJ1bigpIHsgIiRCSU4vbnRsLXNlZ3J1biIgcnVuIC0tbmdsICIkTkdMIiAiJEAiID4+ICIkVC9ydW4ubG9nIiAyPiYxIHx8IHsgZWNobyAiRkFJTEVEOiAkKiI7IHRhaWwgLTMwICIkVC9ydW4ubG9nIjsgZXhpdCAxOyB9OyB9ClM9JFQvc2Vncwpmb3IgQiBpbiAxIDQ7IGRvCiAgcnVuIC0tbW9kZWwgIiRULyRBUkNILW1vZS5nZ3VmIiAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1vdXQgIiRUL3Njb3Jlc19mdWxsX2IkQi5iaW4iIC0tc2VxLWJhdGNoICRCCiAgZm9yIGR0IGluIGYzMiBmMTYgYmYxNjsgZG8KICAgIHJ1biAtLW1vZGVsICIkUy9zZWcwMC1vZi0wMy1MMC00LmdndWYiICAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1vdXQgIiRUL3N0MS5iaW4iIC0tc3RhdGUtZHR5cGUgJGR0IC0tc2VxLWJhdGNoICRCCiAgICBydW4gLS1tb2RlbCAiJFMvc2VnMDEtb2YtMDMtTDQtOC5nZ3VmIiAgLS10b2tlbnMgIiRUL3Rvay5iaW4iIC0tc3RhdGUtaW4gIiRUL3N0MS5iaW4iIC0tb3V0ICIkVC9zdDIuYmluIiAtLXN0YXRlLWR0eXBlICRkdCAtLXNlcS1iYXRjaCAkQgogICAgcnVuIC0tbW9kZWwgIiRTL3NlZzAyLW9mLTAzLUw4LTEyLmdndWYiIC0tdG9rZW5zICIkVC90b2suYmluIiAtLXN0YXRlLWluICIkVC9zdDIuYmluIiAtLW91dCAiJFQvc2NvcmVzX3NlZ18ke2R0fV9iJEIuYmluIiAtLXNlcS1iYXRjaCAkQgogIGRvbmUKICAjIG1pZGRsZSBzZWdtZW50IHJld3JpdGluZyBpdHMgaW5wdXQgc3RhdGUgaW4gcGxhY2UKICBydW4gLS1tb2RlbCAiJFMvc2VnMDAtb2YtMDMtTDAtNC5nZ3VmIiAgLS10b2tlbnMgIiRUL3Rvay5iaW4iIC0tb3V0ICIkVC9zdDEuYmluIiAtLXN0YXRlLWR0eXBlIGYxNiAtLXNlcS1iYXRjaCAkQgogIHJ1biAtLW1vZGVsICIkUy9zZWcwMS1vZi0wMy1MNC04LmdndWYiICAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1zdGF0ZS1pbiAiJFQvc3QxLmJpbiIgLS1vdXQgIiRUL3N0MS5iaW4iIC0tc3RhdGUtZHR5cGUgZjE2IC0tc2VxLWJhdGNoICRCCiAgcnVuIC0tbW9kZWwgIiRTL3NlZzAyLW9mLTAzLUw4LTEyLmdndWYiIC0tdG9rZW5zICIkVC90b2suYmluIiAtLXN0YXRlLWluICIkVC9zdDEuYmluIiAtLW91dCAiJFQvc2NvcmVzX3NlZ19mMTZpbnBsYWNlX2IkQi5iaW4iIC0tc2VxLWJhdGNoICRCCmRvbmUKIyBwYWNrZWQgZGVjb2RlcyAocGFkZGluZywgdW5pZmllZCBLViwgdG9rZW4gYnVkZ2V0KSB3aXRoIGhhbGYtc2l6ZSB1YmF0Y2hlcywgd2hvbGUgYW5kIHNlZ21lbnRlZApydW4gLS1tb2RlbCAiJFQvJEFSQ0gtbW9lLmdndWYiIC0tdG9rZW5zICIkVC90b2suYmluIiAtLW91dCAiJFQvc2NvcmVzX2Z1bGxfcGFja19iNC5iaW4iIC0tc2VxLWJhdGNoIDQgLS1wYWNrIC0tdWJhdGNoLWRpdiAyCnJ1biAtLW1vZGVsICIkUy9zZWcwMC1vZi0wMy1MMC00LmdndWYiICAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1vdXQgIiRUL3N0MS5iaW4iIC0tc3RhdGUtZHR5cGUgZjMyIC0tc2VxLWJhdGNoIDQgLS1wYWNrIC0tdWJhdGNoLWRpdiAyCnJ1biAtLW1vZGVsICIkUy9zZWcwMS1vZi0wMy1MNC04LmdndWYiICAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1zdGF0ZS1pbiAiJFQvc3QxLmJpbiIgLS1vdXQgIiRUL3N0MS5iaW4iIC0tc3RhdGUtZHR5cGUgZjMyIC0tc2VxLWJhdGNoIDQgLS1wYWNrIC0tdWJhdGNoLWRpdiAyCnJ1biAtLW1vZGVsICIkUy9zZWcwMi1vZi0wMy1MOC0xMi5nZ3VmIiAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1zdGF0ZS1pbiAiJFQvc3QxLmJpbiIgLS1vdXQgIiRUL3Njb3Jlc19zZWdfZjMycGFja19iNC5iaW4iIC0tc2VxLWJhdGNoIDQgLS1wYWNrIC0tdWJhdGNoLWRpdiAyCiMgdHdvIGhhbHZlcyBvZiBlYWNoIHNlZ21lbnQgb24gdHdvIGRldmljZXMgKC0tc3BsaXQtbWlkKTogc2FtZSBzY29yZXMsIGFuZCB3aXRoID49IDIgR1BVcyB0aGUgbGF5ZXJzIG9mIHRoZQojIG1pZGRsZSBzZWdtZW50IG11c3QgcmVhbGx5IGJlIHNwbGl0IChmaXJzdCBoYWxmIG9uIENVREEwLCBzZWNvbmQgaGFsZiBvbiBDVURBMSkKOiA+ICIkVC9zcGxpdC5sb2ciCnJ1bnMoKSB7ICIkQklOL250bC1zZWdydW4iIHJ1biAtLW5nbCAiJE5HTCIgLS1zcGxpdC1taWQgIiRAIiA+PiAiJFQvc3BsaXQubG9nIiAyPiYxIHx8IHsgZWNobyAiRkFJTEVEOiAkKiI7IHRhaWwgLTMwICIkVC9zcGxpdC5sb2ciOyBleGl0IDE7IH07IH0KcnVucyAtLW1vZGVsICIkUy9zZWcwMC1vZi0wMy1MMC00LmdndWYiICAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1vdXQgIiRUL3N0MS5iaW4iIC0tc3RhdGUtZHR5cGUgZjMyIC0tc2VxLWJhdGNoIDQKOiA+ICIkVC9zcGxpdF9taWQubG9nIgoiJEJJTi9udGwtc2VncnVuIiBydW4gLS1uZ2wgIiROR0wiIC0tc3BsaXQtbWlkIC0tbW9kZWwgIiRTL3NlZzAxLW9mLTAzLUw0LTguZ2d1ZiIgLS10b2tlbnMgIiRUL3Rvay5iaW4iIC0tc3RhdGUtaW4gIiRUL3N0MS5iaW4iIC0tb3V0ICIkVC9zdDIuYmluIiAtLXN0YXRlLWR0eXBlIGYzMiAtLXNlcS1iYXRjaCA0ID4gIiRUL3NwbGl0X21pZC5sb2ciIDI+JjEgfHwgeyBlY2hvICJGQUlMRUQgc3BsaXQgbWlkIjsgdGFpbCAtMzAgIiRUL3NwbGl0X21pZC5sb2ciOyBleGl0IDE7IH0KcnVucyAtLW1vZGVsICIkUy9zZWcwMi1vZi0wMy1MOC0xMi5nZ3VmIiAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1zdGF0ZS1pbiAiJFQvc3QyLmJpbiIgLS1vdXQgIiRUL3Njb3Jlc19zZWdfZjMyc3BsaXRfYjQuYmluIiAtLXNlcS1iYXRjaCA0CiMgdGhlIGNvbmZpZ3VyYXRpb24gb2YgdGhlIHJlYWwgcnVuczogc3BsaXQgKyBwYWNrICsgaGFsZiB1YmF0Y2hlcyAocGlwZWxpbmUgcGFyYWxsZWwgb24gdHdvIEdQVXMpCnJ1bnMgLS1tb2RlbCAiJFMvc2VnMDAtb2YtMDMtTDAtNC5nZ3VmIiAgLS10b2tlbnMgIiRUL3Rvay5iaW4iIC0tb3V0ICIkVC9zdDEuYmluIiAtLXN0YXRlLWR0eXBlIGYzMiAtLXNlcS1iYXRjaCA0IC0tcGFjayAtLXViYXRjaC1kaXYgMgpydW5zIC0tbW9kZWwgIiRTL3NlZzAxLW9mLTAzLUw0LTguZ2d1ZiIgIC0tdG9rZW5zICIkVC90b2suYmluIiAtLXN0YXRlLWluICIkVC9zdDEuYmluIiAtLW91dCAiJFQvc3QyLmJpbiIgLS1zdGF0ZS1kdHlwZSBmMzIgLS1zZXEtYmF0Y2ggNCAtLXBhY2sgLS11YmF0Y2gtZGl2IDIKcnVucyAtLW1vZGVsICIkUy9zZWcwMi1vZi0wMy1MOC0xMi5nZ3VmIiAtLXRva2VucyAiJFQvdG9rLmJpbiIgLS1zdGF0ZS1pbiAiJFQvc3QyLmJpbiIgLS1vdXQgIiRUL3Njb3Jlc19zZWdfZjMyc3BsaXRwYWNrX2I0LmJpbiIgLS1zZXEtYmF0Y2ggNCAtLXBhY2sgLS11YmF0Y2gtZGl2IDIKZ3JlcCAic3BsaXQtbWlkOiIgIiRUL3NwbGl0X21pZC5sb2ciCk5fR1BVPSQoIChudmlkaWEtc21pIC1MIDI+L2Rldi9udWxsIHx8IHRydWUpIHwgZ3JlcCAtYyAiXkdQVSAiIHx8IHRydWUpCmlmIFsgIiROR0wiICE9ICIwIiBdICYmIFsgIiROX0dQVSIgLWdlIDIgXTsgdGhlbgogICIkUFkiIC1JIC1jICIKaW1wb3J0IHJlLCBzeXMKZGV2ID0gZGljdCgoaW50KGwpLCBkKSBmb3IgbCwgZCBpbiByZS5maW5kYWxsKHInbGF5ZXIgKyhcZCspIGFzc2lnbmVkIHRvIGRldmljZSAoXHcrKScsIG9wZW4oJyRUL3NwbGl0X21pZC5sb2cnKS5yZWFkKCkpKQpnb3QgPSBbZGV2LmdldChpKSBmb3IgaSBpbiByYW5nZSg0LCA4KV0KcHJpbnQoJ3NwbGl0LW1pZCBkZXZpY2VzIG9mIGxheWVycyA0Li43OicsIGdvdCkKc3lzLmV4aXQoMCBpZiBnb3QgPT0gWydDVURBMCcsICdDVURBMCcsICdDVURBMScsICdDVURBMSddIGVsc2UgMSkKIiB8fCB7IGVjaG8gInNwbGl0LW1pZCBkaWQgbm90IHNwbGl0IHRoZSBsYXllcnMiOyBleGl0IDE7IH0KZmkKIiRQWSIgLUkgIiRIRVJFL2NvbXBhcmUucHkiICIkVCIK",
 "slice_gguf.py": "IiIiQ3V0IGEgR0xNNS1OZXh0IEdHVUYgaW50byBsYXllci1yYW5nZSBzZWdtZW50cywgaW4gdGhlIHNhbWUgbGF5b3V0IGFzIHRoZSBwdWJsaWMgS2FnZ2xlIHNlZ21lbnRzCihqaW5naGVuZ3d1L250bC1mbGFzaC1pcTN4eHMtc2Vncyk6IGV2ZXJ5IHNlZ21lbnQga2VlcHMgYWxsIG1ldGFkYXRhLCBob2xkcyBvbmx5IGl0cyBsYXllcnMnIHRlbnNvcnMKYW5kIGFkZHMgbnRsLmxheWVyX3N0YXJ0IC8gbnRsLmxheWVyX2VuZCAvIG50bC5zZWdtZW50X2luZGV4IC8gbnRsLnNlZ21lbnRfY291bnQuCnRva2VuX2VtYmQgZ29lcyBpbiBzZWdtZW50IDAsIG91dHB1dF9ub3JtIGFuZCBvdXRwdXQgaW4gdGhlIGxhc3Qgb25lLiBOZXh0TiAoTVRQKSBsYXllcnMgYXJlIGRyb3BwZWQuCgpPbmx5IHVzZWQgdG8gYnVpbGQgdGhlIHRlc3QgZml4dHVyZSBmb3IgbnRsLXNlZ3J1biAodGhlIHJlYWwgc2VnbWVudHMgYXJlIGFscmVhZHkgcHVibGljKS4KCiAgICBweXRob24gLUkgc3JjL3NlZ3J1bi9zbGljZV9nZ3VmLnB5IE1PREVMLmdndWYgT1VUX0RJUiAwLDMsNiw4CiIiIgppbXBvcnQgb3MKaW1wb3J0IHN5cwoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pLCAnLi4nLCAnLi4nLCAndGhpcmRfcGFydHknLCAnbGxhbWEuY3BwJywgJ2dndWYtcHknKSkKaW1wb3J0IGdndWYgICMgbm9xYTogRTQwMgoKCmRlZiBtYWluKHNyYywgb3V0X2RpciwgYm91bmRzKToKICAgIGJvdW5kcyA9IFtpbnQoeCkgZm9yIHggaW4gYm91bmRzLnNwbGl0KCcsJyldCiAgICByZWFkZXIgPSBnZ3VmLkdHVUZSZWFkZXIoc3JjKQogICAgYXJjaCA9IHJlYWRlci5nZXRfZmllbGQoJ2dlbmVyYWwuYXJjaGl0ZWN0dXJlJykuY29udGVudHMoKQogICAgbl9sYXllciA9IHJlYWRlci5nZXRfZmllbGQoZid7YXJjaH0uYmxvY2tfY291bnQnKS5jb250ZW50cygpCiAgICBuZXh0biA9IHJlYWRlci5nZXRfZmllbGQoZid7YXJjaH0ubmV4dG5fcHJlZGljdF9sYXllcnMnKQogICAgbl90cnVuayA9IG5fbGF5ZXIgLSAobmV4dG4uY29udGVudHMoKSBpZiBuZXh0biBlbHNlIDApCiAgICBhc3NlcnQgYm91bmRzWzBdID09IDAgYW5kIGJvdW5kc1stMV0gPT0gbl90cnVuaywgKGJvdW5kcywgbl90cnVuaykKICAgIG9zLm1ha2VkaXJzKG91dF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICBuX3NlZyA9IGxlbihib3VuZHMpIC0gMQogICAgZm9yIGsgaW4gcmFuZ2Uobl9zZWcpOgogICAgICAgIHMsIGUgPSBib3VuZHNba10sIGJvdW5kc1trICsgMV0KICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKG91dF9kaXIsIGYnc2Vne2s6MDJkfS1vZi17bl9zZWc6MDJkfS1Me3N9LXtlfS5nZ3VmJykKICAgICAgICB3cml0ZXIgPSBnZ3VmLkdHVUZXcml0ZXIocGF0aCwgYXJjaCkKICAgICAgICBmb3IgZmllbGQgaW4gcmVhZGVyLmZpZWxkcy52YWx1ZXMoKToKICAgICAgICAgICAgaWYgZmllbGQubmFtZSA9PSBnZ3VmLktleXMuR2VuZXJhbC5BUkNISVRFQ1RVUkUgb3IgZmllbGQubmFtZS5zdGFydHN3aXRoKCdHR1VGLicpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdnQgPSBmaWVsZC50eXBlc1swXQogICAgICAgICAgICBzdCA9IGZpZWxkLnR5cGVzWy0xXSBpZiB2dCA9PSBnZ3VmLkdHVUZWYWx1ZVR5cGUuQVJSQVkgZWxzZSBOb25lCiAgICAgICAgICAgIGlmIHZ0ID09IGdndWYuR0dVRlZhbHVlVHlwZS5BUlJBWSBhbmQgbGVuKGZpZWxkLmRhdGEpID09IDA6CiAgICAgICAgICAgICAgICBwcmludChmJyAgc2tpcHBpbmcgZW1wdHkgYXJyYXkge2ZpZWxkLm5hbWV9JykKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHdyaXRlci5hZGRfa2V5X3ZhbHVlKGZpZWxkLm5hbWUsIGZpZWxkLmNvbnRlbnRzKCksIHZ0LCBzdWJfdHlwZT1zdCkKICAgICAgICB3cml0ZXIuYWRkX3VpbnQzMignbnRsLmxheWVyX3N0YXJ0JywgcykKICAgICAgICB3cml0ZXIuYWRkX3VpbnQzMignbnRsLmxheWVyX2VuZCcsIGUpCiAgICAgICAgd3JpdGVyLmFkZF91aW50MzIoJ250bC5zZWdtZW50X2luZGV4JywgaykKICAgICAgICB3cml0ZXIuYWRkX3VpbnQzMignbnRsLnNlZ21lbnRfY291bnQnLCBuX3NlZykKICAgICAgICB3cml0ZXIuYWRkX3N0cmluZygnbnRsLnNvdXJjZV9maWxlJywgb3MucGF0aC5iYXNlbmFtZShzcmMpKQoKICAgICAgICBkZWYga2VlcChuYW1lKToKICAgICAgICAgICAgaWYgbmFtZS5zdGFydHN3aXRoKCdibGsuJyk6CiAgICAgICAgICAgICAgICBpbCA9IGludChuYW1lLnNwbGl0KCcuJylbMV0pCiAgICAgICAgICAgICAgICByZXR1cm4gcyA8PSBpbCA8IGUKICAgICAgICAgICAgaWYgbmFtZS5zdGFydHN3aXRoKCd0b2tlbl9lbWJkJyk6CiAgICAgICAgICAgICAgICByZXR1cm4gayA9PSAwCiAgICAgICAgICAgIHJldHVybiBrID09IG5fc2VnIC0gMSAgIyBvdXRwdXRfbm9ybSwgb3V0cHV0CgogICAgICAgIHRlbnNvcnMgPSBbdCBmb3IgdCBpbiByZWFkZXIudGVuc29ycyBpZiBrZWVwKHQubmFtZSldCiAgICAgICAgZm9yIHQgaW4gdGVuc29yczoKICAgICAgICAgICAgd3JpdGVyLmFkZF90ZW5zb3JfaW5mbyh0Lm5hbWUsIHQuZGF0YS5zaGFwZSwgdC5kYXRhLmR0eXBlLCB0LmRhdGEubmJ5dGVzLCB0LnRlbnNvcl90eXBlKQogICAgICAgIHdyaXRlci53cml0ZV9oZWFkZXJfdG9fZmlsZSgpCiAgICAgICAgd3JpdGVyLndyaXRlX2t2X2RhdGFfdG9fZmlsZSgpCiAgICAgICAgd3JpdGVyLndyaXRlX3RpX2RhdGFfdG9fZmlsZSgpCiAgICAgICAgZm9yIHQgaW4gdGVuc29yczoKICAgICAgICAgICAgd3JpdGVyLndyaXRlX3RlbnNvcl9kYXRhKHQuZGF0YSwgdGVuc29yX2VuZGlhbmVzcz1yZWFkZXIuZW5kaWFuZXNzKQogICAgICAgIHdyaXRlci5jbG9zZSgpCiAgICAgICAgcHJpbnQoZid7cGF0aH06IGxheWVycyBbe3N9LCB7ZX0pLCB7bGVuKHRlbnNvcnMpfSB0ZW5zb3JzJykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigqc3lzLmFyZ3ZbMTo0XSkK",
 "compare.py": "IiIiQ29tcGFyZSB0aGUgc2NvcmVzIG9mIHRoZSBzZWdtZW50ZWQgcnVucyBhZ2FpbnN0IHRoZSB3aG9sZS1tb2RlbCBydW4gKHNlZSB2YWxpZGF0ZS5zaCkuIiIiCmltcG9ydCBnbG9iCmltcG9ydCBvcwppbXBvcnQgc3lzCgppbXBvcnQgbnVtcHkgYXMgbnAKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pKQppbXBvcnQgc2VnaW8gICMgbm9xYTogRTQwMgoKCmRlZiBtYWluKGQpOgogICAgcmVmID0gc2VnaW8ucmVhZF9zY29yZXMob3MucGF0aC5qb2luKGQsICdzY29yZXNfZnVsbF9iMS5iaW4nKSkKICAgIFIgPSBucC5jb25jYXRlbmF0ZShyZWYpCiAgICBwcmludChmJ3dob2xlIG1vZGVsOiB7bGVuKHJlZil9IHRleHRzLCB7bGVuKFIpfSBzY29yZWQgdG9rZW5zLCAnCiAgICAgICAgICBmJ21lYW4gbG9ncHJvYiB7Uls6LCAwXS5tZWFuKCk6LjRmfSwgbWVhbiBlbnRyb3B5IHtSWzosIDFdLm1lYW4oKTouNGZ9LCBtZWFuIHJhbmsge1JbOiwgM10ubWVhbigpOi4yZn0nKQogICAgb2sgPSBUcnVlCiAgICBmb3IgcCBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihkLCAnc2NvcmVzXyouYmluJykpKToKICAgICAgICBuYW1lID0gb3MucGF0aC5iYXNlbmFtZShwKVtsZW4oJ3Njb3Jlc18nKTotNF0KICAgICAgICBpZiBuYW1lID09ICdmdWxsX2IxJzoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBTID0gc2VnaW8ucmVhZF9zY29yZXMocCkKICAgICAgICBhc3NlcnQgW2xlbih4KSBmb3IgeCBpbiBTXSA9PSBbbGVuKHgpIGZvciB4IGluIHJlZl0sIG5hbWUKICAgICAgICBTID0gbnAuY29uY2F0ZW5hdGUoUykKICAgICAgICBkbHAgPSBucC5hYnMoU1s6LCAwXSAtIFJbOiwgMF0pLm1heCgpCiAgICAgICAgZGggPSBucC5hYnMoU1s6LCAxXSAtIFJbOiwgMV0pLm1heCgpCiAgICAgICAgZG1lYW4gPSBhYnMoU1s6LCAwXS5tZWFuKCkgLSBSWzosIDBdLm1lYW4oKSkKICAgICAgICBucmFuayA9IGludCgoU1s6LCAzXSAhPSBSWzosIDNdKS5zdW0oKSkKICAgICAgICAjIGYzMiBzdGF0ZXMgbXVzdCBtYXRjaCB0byBmbG9hdCBub2lzZTsgZjE2L2JmMTYgc3RhdGVzIHJvdW5kIHRoZSByZXNpZHVhbCBzdHJlYW0gYmV0d2VlbiBzZWdtZW50cwogICAgICAgIHRvbCA9IDFlLTMgaWYgKCdmMzInIGluIG5hbWUgb3IgbmFtZS5zdGFydHN3aXRoKCdmdWxsJykpIGVsc2UgNWUtMgogICAgICAgIGdvb2QgPSBkbHAgPCB0b2wgYW5kIGRoIDwgdG9sCiAgICAgICAgb2sgJj0gZ29vZAogICAgICAgIHByaW50KGYne25hbWU6MTZzfSBtYXh8ZCBsb2dwcm9ifCB7ZGxwOi4yZX0gIG1heHxkIGVudHJvcHl8IHtkaDouMmV9ICB8ZCBtZWFuIGxvZ3Byb2J8IHtkbWVhbjouMmV9ICAnCiAgICAgICAgICAgICAgZidyYW5rIGNoYW5nZXMge25yYW5rfS97bGVuKFIpfSAgeyJPSyIgaWYgZ29vZCBlbHNlICJGQUlMIn0nKQogICAgcHJpbnQoJ0FMTCBPSycgaWYgb2sgZWxzZSAnU09NRSBGQUlMRUQnKQogICAgc3lzLmV4aXQoMCBpZiBvayBlbHNlIDEpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oc3lzLmFyZ3ZbMV0pCg=="
}


In [ ]:
%%writefile /tmp/ntl/src/segrun/segio.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /tmp/ntl/src/segrun/features.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /tmp/ntl/src/segrun/run_pipeline.py
# Third-party code, copied unchanged from a public Kaggle notebook:
#   Jose Padilla (estopadilla), "0.689 | Real GLM-5.3 Likelihood on 2xT4"
#   https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4
# as carried in lightningv08's "0.702 | Soft Typography + GLM + Qwen3.8-27B LoRA".
# It drives the author's `ntl-segrun` binary (a llama.cpp patch) over layer-segment GGUFs.
# ruff: noqa
"""Score text likelihood by streaming each model segment once per state chunk."""

import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    out = ''.join(lines)
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='JSON list of the pinned segment paths')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    a = ap.parse_args()
    main_run(a)


def main_run(a):

    t_start = time.time()

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(json.loads(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
if MODE != 'probe':
    # ---- the author's sources: checked, then llama.cpp at the tag the patch was written for ----
    for name, sha in SEGRUN_SHA256.items():
        raw = base64.b64decode(SEGRUN_B64[name])
        assert hashlib.sha256(raw).hexdigest() == sha, name
        (SRC / name).write_bytes(raw)
    sh(f'{sys.executable} -I -c "import numpy, yaml"')   # validate.sh runs slice_gguf.py (gguf-py) this way
    if not (LLAMA / '.git').exists():
        sh(f'git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {LLAMA}')
        sh(f'cd {LLAMA} && git apply --check {SRC}/llama-cpp-b12247-ntl.patch && '
           f'git apply {SRC}/llama-cpp-b12247-ntl.patch && git status --short | head -20')
    assert (LLAMA / 'gguf-py' / 'gguf').is_dir(), 'llama.cpp clone failed (internet on?)'

    # ---- build ntl-segrun for this CPU: no CUDA, native instruction set, OpenMP threads ----
    # always name the targets: a bare build also links test-chat/test-server, which need the server
    t = time.time()
    if shutil.which('cmake') is None:
        sh(f'{sys.executable} -m pip install -q cmake ninja')
    CMAKE_FLAGS = ('-DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF -DGGML_NATIVE=ON -DGGML_BLAS=OFF -DGGML_OPENMP=ON '
                   '-DBUILD_SHARED_LIBS=OFF -DLLAMA_BUILD_TESTS=ON -DLLAMA_BUILD_TOOLS=ON -DLLAMA_BUILD_SERVER=OFF '
                   '-DLLAMA_BUILD_EXAMPLES=OFF')
    sh(f'cd {LLAMA} && cmake -S . -B build-cpu {CMAKE_FLAGS} > build.log 2>&1 || (tail -40 build.log; false)')
    sh(f'cd {LLAMA} && cmake --build build-cpu -j{os.cpu_count()} --target ntl-segrun test-llama-archs >> build.log 2>&1 '
       '|| (grep -n -i "error" build.log | head -40; false)')
    for name in ('ntl-segrun', 'test-llama-archs'):
        shutil.copy(LLAMA / 'build-cpu' / 'bin' / name, BIN / name)
        (BIN / name).chmod(0o755)
    deps = sh(f'ldd {BIN}/ntl-segrun', check=False).stdout
    assert not re.search(r'libcu(da|blas|dart)', deps), 'the CPU build links CUDA libraries'
    print(f'build {time.time() - t:.0f} s', flush=True)

    # ---- the author's check: a random model run by layer segments scores the same as run whole ----
    t = time.time()
    for arch in ('glm5-next', 'glm-dsa'):
        r = sh(f'NTL_TEST_DIR={BASE}/validate bash {SRC}/validate.sh {BIN} {sys.executable} 0 {arch}', check=False)
        assert r.returncode == 0 and 'ALL OK' in r.stdout, f'validation failed for {arch}'
    print(f'validation {time.time() - t:.0f} s', flush=True)

    # a record of what was built, kept with the output
    (OUT / 'ntl-bin').mkdir(exist_ok=True)
    shutil.copy(BIN / 'ntl-segrun', OUT / 'ntl-bin' / 'ntl-segrun')
    model_name = re.search(r'^model name\s*:(.*)$', open('/proc/cpuinfo').read(), re.M).group(1).strip()
    (OUT / 'ntl-bin' / 'stamp.json').write_text(json.dumps({
        'llama_tag': LLAMA_TAG,
        'llama_commit': sh(f'git -C {LLAMA} rev-parse HEAD', check=False).stdout.strip(),
        'patch_sha256': SEGRUN_SHA256['llama-cpp-b12247-ntl.patch'],
        'cmake_flags': CMAKE_FLAGS,
        'cpu': model_name,
        'cpu_flags_sha256': hashlib.sha256(' '.join(sorted(FLAGS)).encode()).hexdigest(),
        'ntl_segrun_sha256': hashlib.sha256((BIN / 'ntl-segrun').read_bytes()).hexdigest(),
        'validate': 'ALL OK (glm5-next, glm-dsa)',
    }, indent=1))


In [ ]:
if MODE != 'probe':
    # ---- score the passages: one model at a time, every segment once per state chunk ----
    train = pd.read_csv(TRAIN)
    if MODE == 'timing':
        jobs = [(train.iloc[:TIMING_ROWS], '')]     # the same passages as the GPU timing run
    else:
        # the parts in order; each one's outputs are written as soon as it finishes, so a session that
        # ends early keeps the parts it completed (a model on its own writes nothing until its end)
        jobs = [(train.iloc[k::n], '' if n == 1 else f'_part{k}of{n}') for k, n in SHARDS]
    SETUP_S = time.time() - T0       # checks, build and validation: paid again by every session
    (OUT / 'setup.json').write_text(json.dumps({'setup_s': SETUP_S}))

    stop = False
    for part, suffix in jobs:
      if stop:
        break
      csv = WORK / f'texts{suffix or "_all"}.csv'
      part.to_csv(csv, index=False)
      print(f'{MODE}: {len(part)} passages in {csv}', flush=True)
      for model in MODELS:
        segs = {'flash': FLASH, 'glm53': GLM53}[model]
        tag = model + suffix
        left = SESSION_LIMIT_S - (time.time() - T0)
        # a model is written only at its very end, so start it only if it can finish
        need = 600
        if model in FORECAST:
            fixed, compute = FORECAST[model]
            need = 1.15 * (fixed + compute * len(part) / len(train))
        if need > left - 300:
            print(f'{tag}: not started, it needs about {need / 3600:.1f} h and {left / 3600:.1f} h are left. '
                  'Run it in a session of its own, or as more parts (SHARDS).', flush=True)
            continue
        # f32 state, as in the GPU runs (auto would drop to bf16 when the disk is short)
        cmd = [sys.executable, str(SRC / 'run_pipeline.py'), '--bin', str(BIN), '--segs', json.dumps(segs),
               '--csv', str(csv), '--out', str(OUT), '--tag', tag, '--work', str(WORK / 'state'),
               '--seq-batch', str(SEQ_BATCH), '--state-dtype', 'f32', '--max-text-tokens', str(MAX_TEXT_TOKENS),
               '--prefix', PREFIX, '--threads', str(THREADS)]
        t = time.time()
        # its own process group, so that stopping it also stops ntl-segrun under it, on a timeout or an
        # interrupt alike (otherwise a re-run would race a detached one on the same state file)
        p = subprocess.Popen(cmd, start_new_session=True)
        try:
            code = p.wait(timeout=left - 300)   # stop in time for Kaggle to keep the output
        except BaseException as e:
            os.killpg(p.pid, 9)
            p.wait()
            if not isinstance(e, subprocess.TimeoutExpired):
                raise
            print(f'{tag}: stopped at the session limit after {time.time() - t:.0f} s; nothing written for it. '
                  f'Use more parts (SHARDS) or one model per session.', flush=True)
            stop = True
            break
        assert code == 0, f'{tag}: run_pipeline failed ({code}), see {OUT}/{tag}_run.log'
        timing = json.loads((OUT / f'{tag}_timing.json').read_text())
        print(f"{tag}: {timing['n_done']} texts, {timing['n_rows']} token rows, {len(timing['chunks'])} chunk(s), "
              f"{time.time() - t:.0f} s", flush=True)
        if MODE == 'timing':   # same tokens as the GPU timing run: 5,175 rows over the first 40 passages
            assert timing['n_rows'] == 5175, f"{tag}: {timing['n_rows']} token rows, the GPU run had 5175"


In [ ]:
import re


def forecast(tag, n_texts_full):
    # (load s, compute s) of a full run of one model, from a timing run's log
    timing = json.loads((OUT / f'{tag}_timing.json').read_text())
    log = (OUT / f'{tag}_run.log').read_text(errors='replace')
    done = [(float(a), float(b)) for a, b in re.findall(r'done\. load ([0-9.]+) s, compute ([0-9.]+) s', log)]
    assert done, f'no per-segment timings in {tag}_run.log'
    compute = sum(b for _, b in done)
    # everything but computing stays fixed: loading, process start, state I/O, tokenizing (wall time,
    # which runs 16-27% above the "done." lines)
    fixed = timing['total_s'] - compute
    rows_full = timing['n_rows'] / timing['n_done'] * n_texts_full
    compute_full = compute * rows_full / timing['n_rows']
    print(f"{tag}: {len(done)} segments, fixed {fixed / 60:.1f} min, compute {compute / 60:.1f} min for "
          f"{timing['n_rows']} rows ({timing['n_rows'] * len(done) / max(compute, 1e-9):.0f} rows/s per segment). "
          f"Full run: about {rows_full:.0f} rows, {(fixed + compute_full) / 3600:.2f} h", flush=True)
    return fixed, compute_full


if MODE == 'timing':
    # one plan for all of train.csv: each session pays the setup again, then runs its models one
    # after the other; a part of a model (SHARDS) reads every segment again
    n_full = len(pd.read_csv(TRAIN))
    setup_s = json.loads((OUT / 'setup.json').read_text())['setup_s']
    usable = SESSION_LIMIT_S - 300 - setup_s
    cost = {}
    for model in ('flash', 'glm53'):
        if (OUT / f'{model}_timing.json').exists():
            fixed, compute = forecast(model, n_full)
            cost[model] = (1.15 * fixed, 1.15 * compute)   # margin
    sessions = []
    if len(cost) == 2 and sum(f + c for f, c in cost.values()) <= usable:
        sessions.append({'models': ['flash', 'glm53'], 'shard': [0, 1]})
    else:
        for model, (fixed, compute) in cost.items():
            if fixed >= usable:
                print(f'{model}: loading alone takes {fixed / 3600:.1f} h: this machine cannot run it', flush=True)
                continue
            parts = 1
            while fixed + compute / parts > usable:
                parts += 1
            sessions += [{'models': [model], 'shard': [k, parts]} for k in range(parts)]
    total = sum(sum(cost[m][0] + cost[m][1] / s['shard'][1] for m in s['models']) for s in sessions)
    quota_h = (total + len(sessions) * setup_s) / 3600
    print(f'plan: {len(sessions)} full session(s), about {total / 3600:.1f} h of scoring '
          f'plus {setup_s / 3600:.2f} h of setup each, {quota_h:.1f} h of TPU quota in all:', flush=True)
    for s in sessions:
        print(f"  --mode full --models {','.join(s['models'])} --shard {s['shard'][0]}/{s['shard'][1]}", flush=True)
    if quota_h > 20:
        print(f'WARNING: {quota_h:.1f} h is more than the weekly 20 h TPU quota', flush=True)
    (OUT / 'plan.json').write_text(json.dumps({'sessions': sessions, 'forecast': cost}, indent=1))
for f in sorted(OUT.glob('*_features.csv')):
    print(f.name, len(pd.read_csv(f)), 'passages')
sh(f'ls -la {OUT}; du -sh {OUT}', check=False)
print(f'total {time.time() - T0:.0f} s')
